# B2.8 · Idempotency, replay and rollback

**Function B — Application Security with an AI SDLC → The Harness that Runs the SDLC**  ·  *AI for Security*

Builds on **[B2.7 · Self-improving scaffolds](https://spbreed.github.io/cyber-commons/lessons/B2.7.html)**.

| | |
|---|---|
| Open-source tooling | OpenTelemetry |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The hook

The run failed halfway through. Half the actions happened. Whether that is recoverable was decided when the tools were designed, not now, and 'retry the whole thing' is only safe if somebody made it so.

## 2 · The framework

```
   run fails at step 4 of 7

   steps 1-3 already happened.  retry from the start?
   +-------------------+   +--------------------------+
   | idempotent tool   |   | non-idempotent tool      |
   | safe to repeat    |   | second ticket, second PR |
   +-------------------+   +--------------------------+

   decided when the tools were designed, not during the incident
```

Your agent will do the same thing twice. Retries, restarts, a duplicated webhook,
a loop that lost track — the cause varies and the outcome does not.

Whether that matters depends entirely on the action:

- **Naturally idempotent**: setting a config value, adding a label. Doing it
  twice is doing it once.
- **Accumulating**: posting a comment, sending an email. Twice is noise.
- **Dangerous**: issuing a refund, rotating a credential, scaling a cluster.
  Twice is an incident.

The mechanism is an **idempotency key**: a caller-supplied identifier derived
from the *intent*, so a retry of the same intent lands once. The subtlety is
choosing what goes into the key. Include a timestamp and every retry is a new
operation. Include too little and two genuinely different requests collide.

The second half of this lesson is **replay** — the same property viewed from
forensics. A run you cannot deterministically replay is a run you can describe
but not demonstrate, and D2.5 depends on this being right.

## 3 · Demo — the same action, three natures

In [ ]:
import hashlib, json
from dataclasses import dataclass, field

@dataclass
class Ledger:
    applied: dict = field(default_factory=dict)
    effects: list = field(default_factory=list)

    def apply(self, key, op, **args):
        if key in self.applied:
            self.effects.append(f"SKIP  {op} (key {key[:8]} already applied)")
            return False
        self.applied[key] = (op, args)
        self.effects.append(f"APPLY {op} {args}")
        return True

def idem_key(op, **args):
    """Derived from INTENT. No timestamp, no attempt number."""
    blob = json.dumps({"op": op, "args": args}, sort_keys=True)
    return hashlib.sha256(blob.encode()).hexdigest()

led = Ledger()
# the agent retries the same refund three times
for attempt in range(3):
    k = idem_key("issue_refund", order="ORD-4471", amount=250)
    led.apply(k, "issue_refund", order="ORD-4471", amount=250)
# a genuinely different refund
led.apply(idem_key("issue_refund", order="ORD-4472", amount=90),
          "issue_refund", order="ORD-4472", amount=90)

print("\n".join(led.effects))
print(f"\n4 calls → {len(led.applied)} effects")

## 4 · Where it breaks — a key that includes the wrong thing

In [ ]:
import time

def bad_key_timestamp(op, **args):
    return hashlib.sha256(f"{op}{args}{time.time()}".encode()).hexdigest()

def bad_key_too_narrow(op, **args):
    return hashlib.sha256(op.encode()).hexdigest()          # op only!

led2 = Ledger()
for attempt in range(3):
    led2.apply(bad_key_timestamp("issue_refund", order="ORD-4471", amount=250),
               "issue_refund", order="ORD-4471", amount=250)
print("key includes a timestamp — every retry is a new operation:")
print("\n".join(led2.effects))
print(f"→ refunded {sum(1 for e in led2.effects if e.startswith('APPLY')) * 250} "
      f"instead of 250\n")

led3 = Ledger()
led3.apply(bad_key_too_narrow("issue_refund", order="ORD-4471", amount=250),
           "issue_refund", order="ORD-4471", amount=250)
led3.apply(bad_key_too_narrow("issue_refund", order="ORD-9999", amount=800),
           "issue_refund", order="ORD-9999", amount=800)
print("key includes only the op — different refunds collide:")
print("\n".join(led3.effects))
print("→ the second customer never got their money")

## 5 · The control — classify the action, then key on intent

In [ ]:
ACTIONS = {
 "set_config":     ("naturally idempotent", False),
 "add_label":      ("naturally idempotent", False),
 "post_comment":   ("accumulating",         True),
 "send_email":     ("accumulating",         True),
 "issue_refund":   ("dangerous",            True),
 "rotate_secret":  ("dangerous",            True),
 "scale_cluster":  ("dangerous",            True),
}
print(f"{'action':16s}{'nature':22s}needs a key")
print("-" * 52)
for a, (nature, needs) in ACTIONS.items():
    print(f"{a:16s}{nature:22s}{needs}")

def guarded_call(led, op, **args):
    nature, needs_key = ACTIONS[op]
    if not needs_key:
        led.effects.append(f"APPLY {op} {args} (idempotent by nature)")
        return True
    return led.apply(idem_key(op, **args), op, **args)

led4 = Ledger()
for _ in range(2):
    guarded_call(led4, "set_config", key="tls_min", value="1.2")
    guarded_call(led4, "issue_refund", order="ORD-4471", amount=250)
print("\n" + "\n".join(led4.effects))

In [ ]:
# Verify — replay: the forensic half of the same property.
@dataclass
class Replay:
    prompts: list = field(default_factory=list)
    tool_results: list = field(default_factory=list)
    model_version: str = ""
    seed: object = None
    def replayable(self):
        missing = []
        if not self.prompts:      missing.append("prompts not recorded")
        if not self.tool_results: missing.append("tool results not recorded — "
                                                 "the agent saw a world you cannot rebuild")
        if not self.model_version: missing.append("model version not pinned — "
                                                  "a silent upgrade changes the output")
        if self.seed is None:     missing.append("no seed — sampling makes it unrepeatable")
        return (not missing), missing

for name, r in (("fully instrumented", Replay(["p"], ["tool out"], "glm-4.6@2026-07", 42)),
                ("typical production", Replay(["p"], ["tool out"], "", None)),
                ("actions only",       Replay([], [], "", None))):
    ok, missing = r.replayable()
    print(f"{name:22s} replayable={ok}")
    for m in missing: print(f"      ✗ {m}")
assert Replay(["p"], ["t"], "m", 1).replayable()[0]

## What you just proved

Four refund calls with intent-derived keys produce two effects. A timestamped key refunds 750 instead of 250; an op-only key collides and the second customer is never refunded. The action table marks config and labels as needing no key. The replay check passes only the fully instrumented run, flagging the typical production run for a missing pinned model version and seed.

## Your turn

List the actions your harness can take and mark the dangerous ones. Then check whether each has a key, and what goes into it. A key containing a timestamp or an attempt number is not a key.

---

**Next → [B2.9 · Building a domain harness: one skeleton, four oracles](https://spbreed.github.io/cyber-commons/lessons/B2.9.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.8.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.8.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*